In [2]:
import pandas as pd

df_model_LLM_parse = pd.read_csv("df_model_LLM_parse.csv")
df_final_LLM_parse = pd.read_csv("df_final_LLM_parse.csv")
df_model_LLM_parse = df_model_LLM_parse[:1278].copy()
df_final_LLM_parse = df_final_LLM_parse.copy()

print(df_model_LLM_parse.shape)
print(df_final_LLM_parse.shape)

(1278, 18)
(1278, 20)


In [3]:
from pathlib import Path
from openai import OpenAI


def _parse_yes_no_to_bool(reply_text: str) -> bool:
    """Convert a model reply into a strict boolean (yes=True, no=False)."""
    normalized = (reply_text or "").strip().lower()
    if normalized.startswith("yes"):
        return True
    if normalized.startswith("no"):
        return False
    raise ValueError(f"Unexpected model response: {reply_text!r}")


def _emotion_mentioned(
    text: str,
    client: OpenAI,
    user_prompt_template: str,
    model: str = "chat",
) -> bool:
    """Return True/False by calling the OpenAI-compatible Tulip client.

    Calls `client.chat.completions.create(...)` and extracts the textual reply
    robustly from common response shapes.
    """
    if "{text}" in user_prompt_template:
        user_prompt = user_prompt_template.format(text=text)
    else:
        user_prompt = f"{user_prompt_template}\n\nText:\n{text}"

    system_msg = {"role": "system", "content": "You are a strict binary classifier. Answer with exactly one word: yes or no."}
    user_msg = {"role": "user", "content": user_prompt}

    response = client.chat.completions.create(model=model, messages=[system_msg, user_msg], temperature=0)

    # Extract answer text from several possible response shapes
    answer = ""
    try:
        if isinstance(response, dict):
            # dict-style response
            choices = response.get("choices") or []
            if choices:
                first = choices[0]
                if isinstance(first, dict):
                    msg = first.get("message") or first.get("delta")
                    if isinstance(msg, dict):
                        content = msg.get("content")
                        if isinstance(content, str):
                            answer = content
                    elif isinstance(first.get("text"), str):
                        answer = first.get("text")
        else:
            # SDK-style object
            choices = getattr(response, "choices", None)
            if choices:
                first = choices[0]
                msg = getattr(first, "message", None)
                if msg is not None:
                    content = getattr(msg, "content", None)
                    if isinstance(content, str):
                        answer = content
                text_attr = getattr(first, "text", None)
                if not answer and isinstance(text_attr, str):
                    answer = text_attr

        answer = (answer or "").strip()
    except Exception:
        answer = ""

    return _parse_yes_no_to_bool(answer)


def _read_api_key_from_file(file_name: str = "tulip.key") -> str:
    """Read Tulip API key from a local file in the current working directory."""
    key_path = Path(file_name)
    if not key_path.exists():
        raise FileNotFoundError(
            f"Could not find {file_name} in {Path.cwd()}. Create the file and put your API key in it."
        )

    api_key = key_path.read_text(encoding="utf-8").strip()
    if not api_key:
        raise ValueError(f"{file_name} is empty. Add your API key to this file.")

    return api_key


def add_emotion_flag(
    frame: pd.DataFrame,
    user_prompt_template: str = (
        "Does this text mention any reference to the emotion of the participant "
        "(for example: happy, sad, angry, afraid, anxious, hopeful, "
        "frustrated, proud, etc.)?\n\nText:\n{text}"
    ),
    description_col: str = "intention_description",
    explanation_col: str = "intention_explanation",
    output_col: str = "emotion_mentioned",
    model: str = "chat",
    base_url: str = "https://api.tulip.tudelft.nl/chat/v1/",
) -> pd.DataFrame:
    """
    Loop over every row, apply the prompt to intention_description +
    intention_explanation, and return a dataframe with a boolean output column.
    """
    api_key = _read_api_key_from_file("tulip.key")
    client = OpenAI(base_url=base_url, api_key=api_key)
    result = frame.copy()

    flags = []
    for _, row in result.iterrows():
        description = "" if pd.isna(row.get(description_col)) else str(row.get(description_col))
        explanation = "" if pd.isna(row.get(explanation_col)) else str(row.get(explanation_col))
        combined_text = f"{description}\n\n{explanation}".strip()

        if not combined_text:
            flags.append(False)
            continue

        try:
            flags.append(
                _emotion_mentioned(
                    combined_text,
                    client=client,
                    user_prompt_template=user_prompt_template,
                    model=model,
                )
            )
        except Exception:
            flags.append(False)

    result[output_col] = flags
    return result

In [4]:
import random


def _which_is_human(a_text: str, b_text: str, client: OpenAI, model: str = "chat") -> str:
    system_msg = {
        "role": "system",
        "content": (
            "You are a judge. Given two annotations A and B for the same prompt, "
            "answer with exactly one letter: 'A' if Annotation A was written by a human and Annotation B was written by an LLM; "
            "'B' if Annotation B was written by a human and Annotation A was written by an LLM."
        ),
    }
    user_prompt = f"Annotation A:\n{a_text}\n\nAnnotation B:\n{b_text}\n\nWhich annotation was written by a human? Reply exactly with A or B."
    user_msg = {"role": "user", "content": user_prompt}
    response = client.chat.completions.create(model=model, messages=[system_msg, user_msg], temperature=0)

    answer = ""
    try:
        if isinstance(response, dict):
            choices = response.get("choices") or []
            if choices:
                first = choices[0]
                if isinstance(first, dict):
                    msg = first.get("message") or first.get("delta")
                    if isinstance(msg, dict):
                        content = msg.get("content")
                        if isinstance(content, str):
                            answer = content
                    elif isinstance(first.get("text"), str):
                        answer = first.get("text")
        else:
            choices = getattr(response, "choices", None)
            if choices:
                first = choices[0]
                msg = getattr(first, "message", None)
                if msg is not None:
                    content = getattr(msg, "content", None)
                    if isinstance(content, str):
                        answer = content
                text_attr = getattr(first, "text", None)
                if not answer and isinstance(text_attr, str):
                    answer = text_attr
        answer = (answer or "").strip()
    except Exception:
        answer = ""

    ans_upper = answer.strip().upper()[:1] if answer else ""
    if ans_upper in ("A", "B"):
        return ans_upper
    if "A" in answer and "B" not in answer:
        return "A"
    if "B" in answer and "A" not in answer:
        return "B"
    return "U"


# Run the LLM-as-judge experiment and save results
api_key = _read_api_key_from_file("tulip.key")
client = OpenAI(base_url="https://api.tulip.tudelft.nl/chat/v1/", api_key=api_key)

n = min(len(df_model_LLM_parse), len(df_final_LLM_parse))
rows = []

for i in range(n):
    model_row = df_model_LLM_parse.iloc[i]
    human_row = df_final_LLM_parse.iloc[i]
    m_desc = "" if pd.isna(model_row.get("intention_description")) else str(model_row.get("intention_description"))
    m_exp = "" if pd.isna(model_row.get("intention_explanation")) else str(model_row.get("intention_explanation"))
    h_desc = "" if pd.isna(human_row.get("intention_description")) else str(human_row.get("intention_description"))
    h_exp = "" if pd.isna(human_row.get("intention_explanation")) else str(human_row.get("intention_explanation"))
    model_text = f"{m_desc}\n\n{m_exp}".strip()
    human_text = f"{h_desc}\n\n{h_exp}".strip()
    if not model_text and not human_text:
        continue
    if random.random() < 0.5:
        A_text, B_text, true = human_text, model_text, "A"
    else:
        A_text, B_text, true = model_text, human_text, "B"
    try:
        pred = _which_is_human(A_text, B_text, client=client, model="chat")
    except Exception:
        pred = "U"
    correct = (pred == true)
    rows.append({"index": i, "A_text": A_text, "B_text": B_text, "true": true, "pred": pred, "correct": correct})

res_df = pd.DataFrame(rows)
res_df.to_csv("LLM_as_judge.csv", index=False)
accuracy = res_df["correct"].mean() if not res_df.empty else float("nan")
print("N pairs:", len(res_df))
print("Accuracy:", accuracy)

N pairs: 1278
Accuracy: 0.49139280125195617
